In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 3,
        'suites': [{'cases': [{'code': ">>> type(gamertag) == str and gamertag == 'NovaByte' "
                                       'and type(level) == int and level == 47\n'
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> type(kd_ratio) == float and kd_ratio == 1.85 and '
                                       'is_ranked is True\n'
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> clan is None\nTrue',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 3,
        'suites': [{'cases': [{'code': ">>> q2_a.strip().lower() == 'int'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q2_b.strip().lower() == 'str'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': ">>> q2_c.strip().lower() == 'bool'\nTrue",
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q3_option.strip().upper() == 'C'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> type(q4_level) == int and q4_level == 47\nTrue',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> type(q4_kd) == float and q4_kd == 1.85\nTrue',
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> q4_display == 'Level 47'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q5_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> season_xp(12, 250)\n3000',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> season_xp(3, 100)\n300',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> season_xp(0, 250)\n0',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('03-Variables-Literals-Types', _TESTS)
check = grader.check

print("Ready. 03-Variables-Literals-Types loaded: 6 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Variables, Literals and Types — Player Card

Open any game with a profile screen and you are looking at half a dozen variables. Your
tag is text. Your level is a whole number. Your K/D ratio has a decimal point. Whether
you are ranked this season is yes-or-no. The clan you are in is *nothing at all*, until
you join one.

Python has a different **type** for each of those, and it cares about the difference in
ways that will bite you if you do not. `47` and `"47"` look the same on screen and behave
completely differently.

This homework builds a player card and then pokes at it: what type is each part, what
happens when you convert between them, and what happens when a conversion cannot work.

## Question 1 — Build the card

Create five variables holding exactly these values:

| Variable | Value | Type |
|---|---|---|
| `gamertag` | NovaByte | `str` |
| `level` | 47 | `int` |
| `kd_ratio` | 1.85 | `float` |
| `is_ranked` | yes | `bool` |
| `clan` | not in one yet | `None` |

Three of those are worth watching:

- `level` is `47`, not `"47"` and not `47.0`. A whole number written without a decimal
  point is an `int`.
- `is_ranked` is the Boolean literal `True` — capital T, no quotes. Not `"True"`, which is
  a five-character string, and not `1`, which is an `int`. (Python will tell you `1 == True`
  is `True`, which is a piece of trivia that has misled generations of first-years. They
  are still different types.)
- `clan` is `None`, Python's way of saying "there is deliberately no value here". It is not
  `0`, and it is not the empty string.

Each row of the test is worth 1 point.

In [ ]:
gamertag = ...
level = ...
kd_ratio = ...
is_ranked = ...
clan = ...

print(gamertag, level, kd_ratio, is_ranked, clan)

In [ ]:
grader.check("q1")

## Question 2 — Name the types

The `type()` function tells you what Python thinks a value is:

```python
print(type(47))        # <class 'int'>
```

Work out the type of each of these three, and put the **name** of that type — as a
lower-case string — into `q2_a`, `q2_b` and `q2_c`:

| Variable | The value to examine |
|---|---|
| `q2_a` | `2026` |
| `q2_b` | `"2026"` |
| `q2_c` | `20.26 > 2026` |

So if the answer were a floating point number you would write `q2_a = 'float'`.

The third one is the interesting one: it is an **expression**, not a literal. Python works
out the comparison first, and then you are asking about the type of *the result*. Use the
scratch cell to check yourself — but work it out on paper first, because the exam will not
have a scratch cell.

Each is worth 1 point.

In [ ]:
# a scratch cell to check yourself with
print(type(2026), type("2026"), type(20.26 > 2026))

q2_a = ...
q2_b = ...
q2_c = ...

In [ ]:
grader.check("q2")

## Question 3 — When conversion fails

A web form hands you the text `"12.5"` and you want a whole number, so you write:

```python
int("12.5")
```

What happens?

**A** — you get `12`; Python rounds down

**B** — you get `12.5`

**C** — Python raises a `ValueError`

Set `q3_option` to `'A'`, `'B'` or `'C'`. Run it in a scratch cell and read the message
Python gives you — it names the function and shows the text it could not convert, which is
exactly the information you need to fix it.

(`int()` *will* discard the decimal part of a `float`: `int(12.5)` is `12`. It is the
**string** `"12.5"` it refuses. `int()` on a string wants text that is already a whole
number, nothing else.)

In [ ]:
q3_option = ...

In [ ]:
grader.check("q3")

## Question 4 — Convert between types

Everything typed into a form arrives as text, even when it looks like a number. Here is
the card again, as it comes off the wire:

```python
raw_level = "47"
raw_kd = "1.85"
```

Produce three values from those two strings:

| Variable | What it should hold |
|---|---|
| `q4_level` | the `int` `47` |
| `q4_kd` | the `float` `1.85` |
| `q4_display` | the `str` `'Level 47'` |

For the third one you cannot add a number to a string — `"Level " + 47` is a `TypeError`.
Convert the number back into text with `str()` first, then join them with `+`.

Use the conversion functions on `raw_level` and `raw_kd`; do not just type the answers as
literals.

In [ ]:
raw_level = "47"
raw_kd = "1.85"

q4_level = ...
q4_kd = ...
q4_display = ...

print(q4_level, q4_kd, q4_display)

In [ ]:
grader.check("q4")

## Question 5 — The name Python will not let you use

Which of these three is **not** a legal variable name?

**A** — `player_2`

**B** — `class`

**C** — `_clan`

Set `q5_option` to `'A'`, `'B'` or `'C'`.

Python keeps about thirty words for itself — `if`, `for`, `def`, `return`, `class`, `None`
and friends — and they cannot be used as names. You do not have to memorise the list: try
to assign to one and Python stops with a `SyntaxError` on that line, which is the fastest
error in the language to diagnose once you have seen it once.

While you are here: `player_2` and `_clan` are both legal, but only one of them is a name
you would want to read six months from now. The convention in Python is `snake_case` —
lower case, words joined with underscores, and a name that says what the value *is*.

In [ ]:
q5_option = ...

In [ ]:
grader.check("q5")

## Question 6 — Season XP with a `for` loop

A season runs for a number of weeks, and you earn the same XP each week. Write
`season_xp(weeks, per_week)`, which returns the total.

```
season_xp(12, 250)  ->  3000
season_xp(3, 100)   ->  300
season_xp(0, 250)   ->  0       a season that has not started
```

Yes, you could multiply. **Write the loop anyway** — this is the pattern the whole rest of
the course is built on, and it is worth meeting it while the arithmetic is easy enough to
check in your head.

The design, in plain language, before any Python:

```
set the total to 0
for each week in the season:
    add per_week to the total
give back the total
```

That translates almost word for word. `range(weeks)` gives you one trip round the loop per
week, and `total += per_week` is shorthand for `total = total + per_week`.

Notice what `range(0)` does for the third case — it produces nothing, the body never runs,
and the total stays at 0. You do not need a special case for it.

In [ ]:
def season_xp(weeks, per_week):
    ...

# try it out
print(season_xp(12, 250))

In [ ]:
grader.check("q6")

## Beyond the homework

Not graded:

- Print the whole card as one tidy block. You will want f-strings, which is the next
  topic.
- `is_ranked = 1` passes an `if is_ranked:` test just as well as `True` does. Find out
  which other values Python treats as false — `0`, `""`, `[]`, `None` — and convince
  yourself why `bool("False")` is `True`.
- Add `prestige` to the card and give it a sensible type. Then argue for a different one.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
